# Simulasi Flu Prediction Pipeline
Di dalam pipeline ini, menambahkan proses manipulasi data tiruan yang lebih realistis, mendeteksi nilai yang hilang (missing values), menyamakan skala angka (scaling), dan melakukan evaluasi.

Pustaka (library) Machine Learning
## Scikit-learn 
(atau biasa ditulis sklearn) adalah pustaka (library) paling populer di Python yang digunakan untuk membangun model Machine Learning tradisional.

Pustaka ini sangat disukai karena menyediakan berbagai algoritma siap pakai (seperti Klasifikasi, Regresi, Clustering) serta alat untuk memproses data dan mengevaluasi performa model.

contoh : Scikit-learn untuk memprediksi apakah seorang pasien terkena Flu (1) atau hanya Masuk Angin Biasa / Common Cold (0) berdasarkan 3 gejala utama: Demam Tinggi, Batuk Kering, dan Nyeri Otot.

Menentukan Fitur dan Label
Sebelum menulis kode, kita tentukan dulu bagaimana cara komputer membaca gejalanya menggunakan angka (skala 0 sampai 10, di mana 10 artinya sangat parah):
* Fitur (X): [Skala Demam, Skala Batuk, Skala Nyeri Otot]
* Label (y): 0 = Masuk Angin, 1 = Flu

### Simulasi Model Pipeline

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# =====================================================================
# 1. MEMBUAT DATASET PASIEN (SIMULASI DATAFRAME PANDAS)
# =====================================================================
# Gejala: Demam (°C), Batuk (Skala 1-10), Nyeri Otot (Skala 1-10)
data = {
    'Demam': [36.5, 39.2, 36.8, 38.5, 36.6, 39.0, 38.8, 36.7, np.nan, 39.5], # Ada data kosong (NaN)
    'Batuk': [2, 8, 3, 7, 1, 9, 6, 2, 5, 8],
    'Nyeri_Otot': [1, 9, 2, 7, 1, 8, 8, 2, 3, 9],
    'Diagnosa': [0, 1, 0, 1, 0, 1, 1, 0, 0, 1] # 0: Masuk Angin, 1: Flu
}
df = pd.DataFrame(data)
# Pisahkan Fitur (X) dan Label (y)
X = df[['Demam', 'Batuk', 'Nyeri_Otot']]
y = df['Diagnosa']

# Bagi menjadi Data Training dan Data Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
# =====================================================================
# 2. MEMBANGUN MACHINE LEARNING PIPELINE
# =====================================================================
# Pipeline ini akan otomatis menjalankan 3 langkah berurutan:
# Langkah A: Isi data yang kosong (Imputer) dengan nilai rata-rata (mean)
# Langkah B: Samakan skala data (Scaler) agar satuan Celcius dan Skala 1-10 seimbang
# Langkah C: Klasifikasi menggunakan algoritma Random Forest
# =====================================================================
flu_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])
# =====================================================================
# 3. PROSES TRAINING (Menjalankan Seluruh Alur Pipeline)
# =====================================================================
# Hanya dengan satu perintah .fit(), semua tahapan di atas otomatis berjalan
flu_pipeline.fit(X_train, y_train)
# =====================================================================
# 4. EVALUASI MODEL
# =====================================================================
y_pred = flu_pipeline.predict(X_test)
print("=== PERFORMA MODEL PIPELINE ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred) * 100:.1f}%\n")
print(classification_report(y_test, y_pred, target_names=['Masuk Angin', 'Flu']))

=== PERFORMA MODEL PIPELINE ===
Akurasi: 100.0%

              precision    recall  f1-score   support

 Masuk Angin       1.00      1.00      1.00         1
         Flu       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



### Visualiasi dengan Plotly dari simulasi model

In [2]:
import plotly.graph_objects as go
from sklearn.metrics import precision_score, recall_score, f1_score

# Assuming y_test and y_pred are already defined from previous cells
# Classes: 0 for 'Masuk Angin', 1 for 'Flu'
class_labels = ['Masuk Angin', 'Flu']

# Calculate metrics for each class
precision = [precision_score(y_test, y_pred, pos_label=0), precision_score(y_test, y_pred, pos_label=1)]
recall = [recall_score(y_test, y_pred, pos_label=0), recall_score(y_test, y_pred, pos_label=1)]
f1 = [f1_score(y_test, y_pred, pos_label=0), f1_score(y_test, y_pred, pos_label=1)]

fig = go.Figure()

fig.add_trace(go.Bar(name='Precision', x=class_labels, y=precision))
fig.add_trace(go.Bar(name='Recall', x=class_labels, y=recall))
fig.add_trace(go.Bar(name='F1-Score', x=class_labels, y=f1))

fig.update_layout(
    barmode='group',
    title_text='<b>Metrik Performa Model Pipeline (X_test)</b>',
    xaxis_title='Kelas Diagnosa',
    yaxis_title='Skor',
    yaxis_range=[0, 1.1] # Ensure y-axis goes up to 1 for scores
)

fig.show()

Visualisasi bar chart yang telah ditampilkan menyajikan tiga metrik evaluasi kinerja model penting untuk setiap kelas diagnosa ('Masuk Angin' dan 'Flu'):

1.  **Precision (Presisi):**
    *   **Apa itu:** Presisi mengukur seberapa akurat prediksi positif model. Dengan kata lain, dari semua kasus yang diprediksi oleh model sebagai 'Flu' (atau 'Masuk Angin'), berapa banyak yang sebenarnya benar-benar 'Flu' (atau 'Masuk Angin').
    *   **Interpretasi:** Presisi tinggi menunjukkan bahwa model tidak sering membuat prediksi positif palsu (False Positives). Misalnya, jika presisi untuk 'Flu' adalah 1.0 (100%), berarti setiap kali model memprediksi 'Flu', pasien tersebut memang benar 'Flu'.

2.  **Recall (Sensitivitas/Cakupan):**
    *   **Apa itu:** Recall mengukur kemampuan model untuk menemukan semua kasus positif yang relevan. Dengan kata lain, dari semua pasien yang sebenarnya 'Flu' (atau 'Masuk Angin'), berapa banyak yang berhasil dideteksi dengan benar oleh model.
    *   **Interpretasi:** Recall tinggi menunjukkan bahwa model tidak sering melewatkan kasus positif yang sebenarnya (False Negatives). Misalnya, jika recall untuk 'Flu' adalah 1.0 (100%), berarti model berhasil mendeteksi semua pasien yang sebenarnya menderita 'Flu'.

3.  **F1-Score:**
    *   **Apa itu:** F1-Score adalah rata-rata harmonik dari Precision dan Recall. Ini adalah metrik yang baik untuk digunakan ketika Anda menginginkan keseimbangan antara Precision dan Recall, terutama jika ada ketidakseimbangan kelas (jumlah sampel per kelas tidak sama).
    *   **Interpretasi:** F1-Score yang tinggi (mendekati 1.0) menunjukkan bahwa model memiliki kinerja yang baik dalam hal Precision dan Recall secara bersamaan. Ini adalah indikator performa model yang lebih komprehensif dibandingkan melihat Precision atau Recall saja.


**Bagaimana Melihat Visualisasi Anda:**

*   Setiap kelompok bar mewakili satu metrik (Precision, Recall, F1-Score).
*   Pada setiap kelompok bar, Anda akan melihat dua bar: satu untuk 'Masuk Angin' dan satu untuk 'Flu'.
*   **Nilai 1.0 (atau 100%) untuk semua metrik (Precision, Recall, F1-Score) di kedua kelas** menunjukkan bahwa model Anda memiliki performa yang sempurna pada data `X_test` yang digunakan. Ini berarti model berhasil memprediksi semua kasus 'Masuk Angin' dengan benar dan semua kasus 'Flu' dengan benar di set data pengujian tersebut.

In [3]:
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

# Calculate the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Define class labels
class_labels = ['Masuk Angin', 'Flu']

# Create the heatmap using Plotly Graph Objects
fig = go.Figure(data=go.Heatmap(
                   z=cm,
                   x=class_labels,
                   y=class_labels,
                   colorscale='Viridis')) # You can choose other colorscales like 'Blues', 'Greens', etc.

fig.update_layout(
    title_text='<b>Confusion Matrix Model Pipeline</b>',
    xaxis_title='Prediksi',
    yaxis_title='Aktual',
    xaxis_nticks=len(class_labels),
    yaxis_nticks=len(class_labels),
    xaxis_showgrid=False,
    yaxis_showgrid=False
)

# Add text annotations for the values inside the heatmap cells
for i in range(len(class_labels)):
    for j in range(len(class_labels)):
        fig.add_annotation(
            x=class_labels[j],
            y=class_labels[i],
            text=str(cm[i, j]),
            showarrow=False,
            font=dict(color="white" if cm[i,j] > cm.max()/2 else "black", size=14)
        )

fig.show()

Heatmap ini menunjukkan perbandingan antara nilai aktual dan prediksi model untuk 'Masuk Angin' dan 'Flu'

### Simulasi Prediksi dengan Model Pipeline

In [4]:
# =====================================================================
# 5. PREDIKSI PASIEN BARU LEWAT PIPELINE
# =====================================================================
# Pasien Baru datang dengan Demam 38.9°C, Batuk skala 8, Nyeri Otot skala 7
pasien_baru = pd.DataFrame([{
    'Demam': 38.9,
    'Batuk': 8,
    'Nyeri_Otot': 7
}])

# Data pasien baru otomatis di-impute dan di-scale oleh pipeline sebelum diprediksi
prediksi_final = flu_pipeline.predict(pasien_baru)
probabilitas = flu_pipeline.predict_proba(pasien_baru)

print("=== HASIL DIAGNOSA PASIEN BARU ===")
if prediksi_final[0] == 1:
    print(f"Diagnosa: POSITIF FLU (Keyakinan: {probabilitas[0][1]*100:.1f}%)")
else:
    print(f"Diagnosa: NEGATIF FLU / MASUK ANGIN (Keyakinan: {probabilitas[0][0]*100:.1f}%)")

=== HASIL DIAGNOSA PASIEN BARU ===
Diagnosa: POSITIF FLU (Keyakinan: 98.0%)


### Form inputan prediksi diagnosa flu

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import plotly.graph_objects as go

# Mendapatkan nama kolom fitur dari data pelatihan flu_pipeline
feature_columns_flu = ['Demam', 'Batuk', 'Nyeri_Otot']

# Membuat widget input untuk setiap fitur
input_widgets_flu = {}
input_widgets_flu['Demam'] = widgets.FloatSlider(
    value=37.0,
    min=35.0,
    max=41.0,
    step=0.1,
    description='Demam (°C):',
    continuous_update=False
)
input_widgets_flu['Batuk'] = widgets.IntSlider(
    value=5,
    min=0,
    max=15,
    step=1,
    style={'description_width': 'initial'},
    description='Batuk (Skala 0-15):',
    continuous_update=False
)
input_widgets_flu['Nyeri_Otot'] = widgets.IntSlider(
    value=5,
    min=0,
    max=15,
    step=1,
    style={'description_width': 'initial'},
    description='Nyeri Otot (Skala 0-15):',
    continuous_update=False
)

# Tombol prediksi
predict_button_flu = widgets.Button(description="Prediksi Diagnosa Flu")

# Area output untuk menampilkan hasil prediksi dan visualisasi
output_prediction_flu = widgets.Output()
output_plot_flu = widgets.Output()

def on_predict_button_clicked_flu(b):
    with output_prediction_flu:
        clear_output()
        new_patient_input_flu = {feature: widget.value for feature, widget in input_widgets_flu.items()}

        print("Data Pasien Baru dari Input Form:")
        for k, v in new_patient_input_flu.items():
            print(f"  {k.replace('_', ' ').title()}: {v}")

        # Convert new_patient_input to DataFrame for prediction with flu_pipeline
        new_patient_df_flu = pd.DataFrame([new_patient_input_flu], columns=feature_columns_flu)

        # Perform prediction using the flu_pipeline
        prediction_flu = flu_pipeline.predict(new_patient_df_flu)
        prediction_proba_flu = flu_pipeline.predict_proba(new_patient_df_flu)

        result_text = ""
        if prediction_flu[0] == 1:
            result_text = f"Pasien diprediksi **FLU** (Probabilitas: {prediction_proba_flu[0][1]*100:.2f}%)"
        else:
            result_text = f"Pasien diprediksi **MASUK ANGIN** (Probabilitas: {prediction_proba_flu[0][0]*100:.2f}%)"

        print(f"\nHasil Prediksi: {result_text}")

    with output_plot_flu:
        clear_output()
        # Create a DataFrame for probabilities
        proba_df_flu = pd.DataFrame({
            'Class': ['Masuk Angin', 'Flu'],
            'Probability': prediction_proba_flu[0]
        })

        # Create an interactive bar chart for prediction probabilities
        fig_proba_flu = go.Figure(data=[go.Bar(
            x=proba_df_flu['Class'],
            y=proba_df_flu['Probability'],
            marker_color=['green', 'red'] if prediction_flu[0] == 0 else ['red', 'green']
        )])

        fig_proba_flu.update_layout(
            title='<b>Probabilitas Prediksi Diagnosa Flu</b>',
            xaxis_title='Kondisi',
            yaxis_title='Probabilitas',
            yaxis_range=[0, 1.05],
            template='plotly_white'
        )
        fig_proba_flu.show()

predict_button_flu.on_click(on_predict_button_clicked_flu)

# Tampilkan semua widget
print("Masukkan Gejala Pasien Baru untuk Diagnosa Flu:")
for feature in feature_columns_flu:
    display(input_widgets_flu[feature])
display(predict_button_flu, output_prediction_flu, output_plot_flu)

Masukkan Gejala Pasien Baru untuk Diagnosa Flu:


FloatSlider(value=37.0, continuous_update=False, description='Demam (°C):', max=41.0, min=35.0)

IntSlider(value=5, continuous_update=False, description='Batuk (Skala 0-15):', max=15, style=SliderStyle(descr…

IntSlider(value=5, continuous_update=False, description='Nyeri Otot (Skala 0-15):', max=15, style=SliderStyle(…

Button(description='Prediksi Diagnosa Flu', style=ButtonStyle())

Output()

Output()